# 01 — Data Ingestion & Quality Control

**Pipeline Step 1 of 5**

This notebook is the entry point of the Spatial-MicroCKG pipeline. It downloads a publicly available 10x Genomics Visium spatial transcriptomics dataset of the adult mouse brain and applies a standard quality-control (QC) workflow before saving the preprocessed data for downstream analysis.

Spatial transcriptomics captures gene expression while preserving the physical location of each measurement spot within the tissue section. The Visium platform tiles the tissue with barcoded spots (55 µm diameter, ~100 µm center-to-center), each capturing mRNA from a small neighborhood of cells. This spatial information is critical for later steps where we overlay biomarker expression on the histological (H&E-stained) tissue image to validate biological relevance.

The QC step removes low-quality spots and rarely detected genes, ensuring that downstream feature selection (Stabl) operates on a clean, high-confidence expression matrix. After filtering, the data is normalized to make expression values comparable across spots and log-transformed to stabilize variance.

### Outputs
| File | Description |
|---|---|
| `data/processed/mouse_brain_preprocessed.h5ad` | QC-filtered, normalized AnnData object ready for feature selection |

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_ingestion import get_dataset
from src.spatial_pipeline import load_adata, run_qc, normalize

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print("Imports ready.")

Imports ready.


## 1.1 Download Dataset

We use the **squidpy** library to programmatically download the 10x Visium V1 Adult Mouse Brain dataset. This dataset contains approximately 2,700 tissue spots, each profiling the expression of over 32,000 genes. The data also includes an H&E-stained tissue image and spatial coordinates for each spot, enabling anatomical alignment of molecular data.

The dataset is cached locally so re-runs skip the download step. The resulting file is stored in AnnData (`.h5ad`) format, the de facto standard for single-cell and spatial transcriptomics data in the Python ecosystem.

In [2]:
h5ad_path = get_dataset(DATA_RAW, source="squidpy")
adata = load_adata(h5ad_path)
print(f"\nRaw dataset: {adata.shape[0]} spots × {adata.shape[1]} genes")

/Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  Dataset already cached: /Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/data/raw/mouse_brain_visium.h5ad
  Loading dataset: /Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/data/raw/mouse_brain_visium.h5ad
  Shape: 2702 spots × 32285 genes

Raw dataset: 2702 spots × 32285 genes


## 1.2 Quality Control

Quality control is essential to remove technical artifacts before biological analysis. We apply three standard filters:

1. **Minimum genes per spot (≥ 200):** Spots that detect very few genes are likely empty or damaged. Removing them prevents noise from dominating downstream analyses.
2. **Minimum cells per gene (≥ 3):** Genes detected in fewer than three spots provide insufficient statistical evidence and are excluded.
3. **Maximum mitochondrial gene percentage (< 30%):** Elevated mitochondrial RNA content is a hallmark of stressed or dying cells. In brain tissue, mitochondrial fractions are naturally higher than in most other tissues due to the high metabolic demand of neurons, so we use a relaxed threshold of 30% (compared to the typical 5-10% for non-neuronal tissues).

After filtering, you can expect a modest reduction in the number of spots (typically losing only spots from the tissue periphery or low-quality areas) while retaining the vast majority of genes.

In [3]:
adata = run_qc(adata)
print(f"\nPost-QC: {adata.shape[0]} spots × {adata.shape[1]} genes")

  QC filtering: 2702 → 2691 spots
  Genes retained: 19652

Post-QC: 2691 spots × 19652 genes


## 1.3 Normalization

After QC filtering, we apply two normalization steps:

1. **Library-size normalization (CPM-like):** Each spot's total UMI count is scaled to a common target of 10,000 counts. This corrects for variation in sequencing depth between spots, making expression values comparable.
2. **Log-transformation (`log1p`):** A `log(x + 1)` transform is applied to stabilize variance and compress the dynamic range. Highly expressed genes (e.g., ribosomal, mitochondrial) would otherwise dominate variance-based analyses.

Raw counts are preserved in `adata.raw` so that they remain available for differential expression and spatial plotting (which expect un-normalized or separately normalized values).

In [4]:
adata = normalize(adata)
print(f"\nNormalized: {adata.shape[0]} spots × {adata.shape[1]} genes")
print(f"Raw layer preserved: {adata.raw is not None}")

  Normalized to 10000 counts/cell and log1p-transformed

Normalized: 2691 spots × 19652 genes
Raw layer preserved: True


## 1.4 Save Preprocessed Data

The preprocessed AnnData is serialized to disk in `.h5ad` format. This checkpoint allows downstream notebooks (Stabl feature selection, visualization, KG construction) to load a clean, ready-to-use dataset without repeating the QC and normalization steps. This modular design ensures reproducibility and makes it easy to swap in a different dataset at this step.

In [5]:
out_path = DATA_PROCESSED / "mouse_brain_preprocessed.h5ad"
adata.write_h5ad(out_path)
print(f"Saved preprocessed AnnData to {out_path}")
print(f"File size: {out_path.stat().st_size / 1e6:.1f} MB")

Saved preprocessed AnnData to /Users/shaunfchen/Documents/Repositories/spatial-microckg-agent/data/processed/mouse_brain_preprocessed.h5ad
File size: 274.5 MB
